In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import warnings
warnings.filterwarnings('ignore')
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

# JYDB

聚源因子库(JYDB) 是基于关系数据库构建的因子库

基于关系数据库构建的因子库和数据库原始对象的对应关系：
* 整个数据库对应于因子库
* 每张数据库表对应于因子表
* 每张数据库表的字段对应于单个因子

本质上是将一个二维的因子数据矩阵挤压成具有二重索引的一维向量. 对于因子数据的访问, 内部使用标准的 SQL 查询语句完成. 

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库]
        B1[因子表]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子1]
        F2[因子2]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[关系数据库]
        D1[数据库表]
        D2[数据库表]
        C --> D1
        C --> D2
        E2[字段2]
        E1[字段1]
        D2 --> E1
        D2 --> E2
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E1

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
```

以下代码要求有可以访问的聚源数据库，且设置好了配置文件。或者执行 [import_postgres_jydb_demo_data.py](../tools/import_postgres_jydb_demo_data.py) 脚本生成示例数据，这要求有写入权限的 postgresql 数据库。

聚源因子库的配置文件默认位于用户目录下的 “QuantStudioConfig” 文件夹里的 "JYDBConfig.json" 文件。通常将数据库的连接信息配置到文件里，示例如下:
```json
{
    "Name": "JYDB",
    "DBType": "PostgreSQL",
    "DBName": "JYDB",
    "IPAddr": "localhost",
    "Port": 5432,
    "User": "postgres",
    "Pwd": "123456",
    "TablePrefix": "",
    "CharSet": "utf8",
    "Connector": "default"
}
```

In [3]:
# 创建因子库对象并 connect
from QuantStudio.Factor.JYDB import JYDB

FDB = JYDB().connect()
print(qs_help(FDB))

类型: JYDB
模块: QuantStudio.Factor.JYDB
QS 对象类型: 因子库
QS 对象名称: JYDB
QSID: a4602c3052c0032aa821b2f2c4db8e989d51e33a046ece1e6b39f92f1d34c0a2
参数集:
    * Name(名称): <class 'str'>, 默认值 'JYDB', 当前取值: 'JYDB'
    * DBType(数据库类型): typing.Literal['MySQL', 'SQL Server', 'Oracle', 'PostgreSQL'], 默认值 'MySQL', 当前取值: 'PostgreSQL'
    * DBName(数据库名): <class 'str'>, 默认值 'Scorpion', 当前取值: 'JYDB'
    * IPAddr(IP地址): <class 'str'>, 默认值 '127.0.0.1', 当前取值: 'localhost'
    * Port(端口): <class 'int'>, 默认值 3306, 当前取值: 5432
    * User(用户名): <class 'str'>, 默认值 'root', 当前取值: 'shzq'
    * TablePrefix(表名前缀): <class 'str'>, 默认值 '', 当前取值: ''
    * CharSet(字符集): typing.Literal['utf8', 'utf8mb4', 'gbk', 'gb2312', 'gb18030', 'cp936', 'big5'], 默认值 'utf8', 当前取值: 'utf8'
    * Connector(连接器): typing.Literal['default', 'cx_Oracle', 'pymssql', 'mysql.connector', 'pymysql', 'psycopg2', 'pyodbc'], 默认值 'default', 当前取值: 'default'
    * ConnRetryNum(连接重试次数): <class 'int'>, 默认值 3, 当前取值: 3
    * ConnIntervalSeconds(连接重试间隔): <class 'float'

In [4]:
# 获取因子库中的因子表列表
print(FDB.TableNames[:5])

['交易日表(新)', '人员表', '行业表', '行业类别表', 'A股证券主表']


### 获取时点序列

#### 获取交易日序列

In [5]:
# 获取交易日序列的方法说明
print(qs_help(FDB.getTradeDay))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getTradeDay(start_date: Optional[datetime.datetime] = None, end_date: Optional[datetime.datetime] = None, exchange: Literal['SSE', 'SZSE', 'SHFE', 'DCE', 'CZCE', 'INE', 'CFFEX'] = 'SSE', **kwargs) -> List[datetime.datetime]
说明文档:
    给定交易所、起始日和结束日, 获取交易日序列
    
    Args:
        start_date: 起始日, None 表示从 1900-01-01 开始
        end_date: 结束日, None 表示当前日期
        exchange: 交易所, 默认 SSE(上交所)
    
    Returns:
        交易日序列


In [6]:
# 给定起止时点, 获取交易日序列
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 5))
print(DTs)

[datetime.datetime(2022, 1, 4, 0, 0), datetime.datetime(2022, 1, 5, 0, 0)]


### 获取证券代码序列

#### 股票证券代码

In [7]:
# 获取股票证券代码方法说明
print(qs_help(FDB.getStockID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getStockID(exchange: Union[Literal['SSE', 'SZSE', 'BSE', 'HKEX', 'AMEX', 'NASDAQ', 'NYSE', 'NEEQ'], Tuple[Literal['SSE', 'SZSE', 'BSE', 'HKEX', 'AMEX', 'NASDAQ', 'NYSE', 'NEEQ']]] = ('SSE', 'SZSE', 'BSE'), date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定交易所和日期, 获取股票证券 ID 序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 ("SSE", "SZSE", "BSE") 表示上交所、深交所、北交所
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日期在指定日之前的股票, True 表示上市日期在指定日之前且尚未退市的股票
        start_date: 起始日, 如果非 None 并且 is_current=False 表示提取在 start_date 至 date 之间上市过的股票, 如果 is_current=True 表示提取在 start_date 至 date 之间均保持上市的股票
    
    Returns:
        股票证券 ID 序列


In [8]:
# 获取全体A股，包括已经退市的
IDs = FDB.getStockID(is_current=False)
print(IDs[:5])

['000001.SZ', '000002.SZ', '000003.SZ', '000004.SZ', '000005.SZ']


#### 公募基金证券代码

In [9]:
# 获取公募基金证券代码方法说明
print(qs_help(FDB.getMutualFundID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getMutualFundID(type: Optional[Literal['ETF', 'LOF', 'FOF', 'QDII', '封闭基金', 'ETF联接基金', '指数基金', '指数增强基金']] = None, exchange: Union[str, Tuple[str], NoneType] = None, date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定日期, 获取公募基金 ID 序列
    
    Args:
        type: 基金类型, None 表示取所有的基金
        exchange: 交易所(str)或者交易所列表(tuple), 如果非 None 表示只考虑在这些指定的交易所上市的基金, None 表示包括非上市基金
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示成立日在指定日之前的基金, True 表示成立日在指定日之前且尚未清盘的基金
    
    Returns:
        公募基金证券 ID 序列


In [10]:
# 获取全体公募基金，包括已经清盘的
IDs = FDB.getMutualFundID(is_current=False)
print(IDs[:5])

['000001.OF', '000003.OF', '000004.OF', '000005.OF', '000006.OF']


#### 期货证券代码

In [11]:
# 获取期货证券代码方法说明
print(qs_help(FDB.getFutureID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getFutureID(exchange: Union[str, Tuple[str], NoneType] = None, future_code: Union[str, Tuple[str], NoneType] = None, date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定交易所、期货品种代码和日期, 获取期货证券 ID 序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 None 表示所有交易所
        future_code: 期货品种代码(str)或者期货品种代码列表(tuple), None 表示所有期货品种代码
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日在指定日之前的期货, True 表示上市日在指定日之前且尚未退市的期货
        start_date: 起始日, 如果非 None 并且 is_current=False 表示提取在 start_date 至 date 之间上市过的期货, 如果 is_current=True 表示提取在 start_date 至 date 之间均保持上市的期货
        kwargs:
            contract_type: 合约类型, 可选 "月合约", "连续合约", "所有", 默认值 "月合约"
            continue_contract_type: 连续合约类型, list[str], 可选 "主力合约", "期货指数", "次主力合约", "连续合约", "连一合约", "连二合约", "连三合约", "连四合约", "当月连续合约", "次月连续合约", "当季连续合约", "下季连续合约", "隔季连续合约", 默

In [12]:
# 获取所有中国股指期货代码, 包括已经退市的
IDs = FDB.getFutureID(exchange="CFFEX", future_code="IF", is_current=False)
print(IDs[:5])

['IF1005.CFE', 'IF1006.CFE', 'IF1007.CFE', 'IF1008.CFE', 'IF1009.CFE']


#### 期货品种代码

In [13]:
# 获取期货品种代码方法说明
print(qs_help(FDB.getFutureCode))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getFutureCode(exchange: Union[str, Tuple[str], NoneType] = None, date: Optional[datetime.datetime] = None, is_current: bool = True, **kwargs) -> List[str]
说明文档:
    给定交易所和日期, 获取期货品种代码序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 None 表示所有交易所
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日在指定日之前的期货品种, True 表示上市日在指定日之前且尚未退市的期货品种
    
    Returns:
        期货品种代码序列


In [14]:
# 获取所有中国期货品种代码, 包括已经退市的
IDs = FDB.getFutureCode(exchange=("SHFE", "INE", "DCE", "CZCE", "GFEX", "CFFEX"), is_current=False)
print(IDs[:5])

['A', 'AD', 'AG', 'AL', 'AO']


#### 期权证券代码

In [15]:
# 获取期权品种代码方法说明
print(qs_help(FDB.getOptionID))

类型: method (bound to JYDB)
模块: QuantStudio.Factor.JYDB
签名: JYDB.getOptionID(option_code: str = '510050', date: Optional[datetime.datetime] = None, is_current: bool = True, start_date: Optional[datetime.datetime] = None, **kwargs) -> List[str]
说明文档:
    给定期权品种代码和日期, 获取期权证券 ID 序列
    
    Args:
        option_code: 期权品种代码
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日在指定日之前的期权, True 表示上市日在指定日之前且尚未退市的期权
    
    Returns:
        期权证券 ID 序列


In [16]:
# 获取所有上证50ETF期权代码, 包括已经退市的
IDs = FDB.getOptionID(option_code="510050", is_current=False)
print(IDs[:5])

['510050C1503M02200', '510050C1503M02250', '510050C1503M02300', '510050C1503M02350', '510050C1503M02400']


# 因子表

In [17]:
# 获取因子表对象
FT = FDB.getTable("日行情表", args={"LookBack": 0})
print(qs_help(FT))

类型: _WideTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 日行情表
QSID: a56c7bd51fb7de8ea1c14d36ca61e71ea5f8af32699e8eb687568345841ed354
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '日行情表'
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'WideTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '交易日'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 0 表示不回溯填充, 当前取值: 0
    * PublDTField(公告时点字段): typing.Optional[str], 默认值 None, 用作公告时点的字段名, 默认值 None 表示内部自动判断, 如果非 None, 表示考虑数据的公布时点, 即某个时点所能获取的数据必须保证其在公告时点和截止时点之后, 当前取值: None
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 apply 的函数 f(x), 其中 x 为 Series, 默认值 None 表示使用 lambda x: x.

In [18]:
# 因子列表
FT = FDB.getTable("日行情表")
print(FT.FactorNames)

['证券内部编码', '交易日', '昨收盘(元)', '今开盘(元)', '最高价(元)', '最低价(元)', '收盘价(元)', '成交量(股)', '成交金额(元)', '成交笔数(笔)']


## 读取数据

In [19]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

FT = FDB.getTable("日行情表", args={"LookBack": 0})
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

因子表数据
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: 收盘价(元) to 今开盘(元)
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


# 因子

In [20]:
# 获取因子对象
F = FT.getFactor("收盘价(元)")
print(qs_help(F))

类型: Factor
模块: QuantStudio.Factor.Factor
QS 对象类型: 计算节点-因子
QS 对象名称: 收盘价(元)
QSID: 5477475533de21c4ba7ba279d93df55654560ec482b4c8e885885f0210e3883e
参数集:
    * Name(名称): <class 'str'>, 默认值 'Factor', 当前取值: '收盘价(元)'
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
说明文档:
    因子对象
    因子可看做 DataFrame(index=[时点], columns=[ID])
    时点数据类型是 datetime, ID 的数据类型是 str


## 读取数据

In [21]:
# 因子读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = F.readData(ids=IDs, dts=DTs)
print(Data)

            000001.SZ  000002.SZ
2025-01-01        NaN        NaN
2025-01-02      11.43       7.11
2025-01-03      11.38       7.00
2025-01-04        NaN        NaN
2025-01-05        NaN        NaN


# 因子表参数详解

因子表对象有个参数 `TableType` 标识了因子表的类型，不同类型的因子表将数据库中原始数据映射成因子数据的方式不同。下面的子章节根据不同的类型说明其可调参数的含义。

In [22]:
# 因子表类型
FT = FDB.getTable("日行情表", args={"LookBack": 0})
print(FT.Args.TableType)

WideTable


## WideTable

数据库数据示例：

![Wide_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_Demo.png)

In [23]:
FT = FDB.getTable("日行情表")
print(qs_help(FT))

类型: _WideTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 日行情表
QSID: a56c7bd51fb7de8ea1c14d36ca61e71ea5f8af32699e8eb687568345841ed354
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '日行情表'
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'WideTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '交易日'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 0, 缺失填充回溯的天数, 0 表示不回溯填充, 当前取值: 0
    * PublDTField(公告时点字段): typing.Optional[str], 默认值 None, 用作公告时点的字段名, 默认值 None 表示内部自动判断, 如果非 None, 表示考虑数据的公布时点, 即某个时点所能获取的数据必须保证其在公告时点和截止时点之后, 当前取值: None
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 apply 的函数 f(x), 其中 x 为 Series, 默认值 None 表示使用 lambda x: x.

### 缺失填充

In [24]:
# WideTable 不填充缺失, LookBack = 0
FT = FDB.getTable("日行情表", args={"LookBack": 0})

DTs = [dt.datetime(2023, 12, 29) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

            收盘价(元)  今开盘(元)
2023-12-29    9.39    9.42
2023-12-30     NaN     NaN
2023-12-31     NaN     NaN
2024-01-01     NaN     NaN
2024-01-02    9.21    9.39


In [25]:
# WideTable 填充缺失, 最大回溯期 LookBack > 0, 如果为 inf 表示无限制的回溯填充
FT = FDB.getTable("日行情表", args={"LookBack": 2})

DTs = [dt.datetime(2023, 12, 29) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ"]
Data = FT.readData(factor_names=["收盘价(元)", "今开盘(元)"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

            收盘价(元)  今开盘(元)
2023-12-29    9.39    9.42
2023-12-30    9.39    9.42
2023-12-31    9.39    9.42
2024-01-01     NaN     NaN
2024-01-02    9.21    9.39


### 公告时点

数据库数据示例：

![Wide_Table_AnnDT_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_AnnDT_Demo.png)

In [26]:
DTs = [dt.datetime(2023, 6, 30), dt.datetime(2023, 9, 30), dt.datetime(2023, 10, 1)]
IDs = ["000001.SZ"]

# 原始数据
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": 0,
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

# 不考虑公告时点, 下面例子中 2023-10-01 使用了 2023-09-30 的数据进行了填充
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": np.inf,
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("-" * 10)
print(Data)

# 考虑公告时点, 下面例子中 2023-10-01 使用了 2023-06-30 的数据进行了填充
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": np.inf,
    "PublDTField": "信息发布日期",
    "AdditionalCondition": {"是否合并": "1", "是否调整": "2"}
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print("-" * 10)
print(Data)

                         信息发布日期             资产总计
2023-06-30  1692835200000000000  5500524000000.0
2023-09-30  1698192000000000000  5516388000000.0
2023-10-01                  NaN              NaN
----------
                         信息发布日期             资产总计
2023-06-30  1692835200000000000  5500524000000.0
2023-09-30  1698192000000000000  5516388000000.0
2023-10-01  1698192000000000000  5516388000000.0
----------
                         信息发布日期             资产总计
2023-06-30  1682380800000000000  5455897000000.0
2023-09-30  1692835200000000000  5500524000000.0
2023-10-01  1692835200000000000  5500524000000.0


### 多重映射

数据库数据示例：

![Wide_Table_MultiMapping_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Wide_Table_MultiMapping_Demo.png)

In [27]:
# 对于一个报告期，可能有多份财报（包括修正），因此 MultiMapping=True 时一个时点下对应着多个值
DTs = [dt.datetime(2023, 9, 30), dt.datetime(2023, 12, 31)]
IDs = ["000001.SZ"]

FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": 0,
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1"},
    "MultiMapping": True
})
Data = FT.readData(factor_names=["信息发布日期", "资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

# 可以通过指定算子，将多个值合并成一个值
FT = FDB.getTable("资产负债表_新会计准则", args={
    "TableType": "WideTable",
    "LookBack": 0,
    "PublDTField": None,
    "AdditionalCondition": {"是否合并": "1"},
    "MultiMapping": True,
    "Operator": lambda x: x.mean(),
    "OperatorDataType": "double"
})
Data = FT.readData(factor_names=["资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

                                                                信息发布日期  \
2023-09-30 00:00:00                              [2023-10-25 00:00:00]   
2023-12-31 00:00:00  [2024-03-15 00:00:00, 2024-04-20 00:00:00, 202...   

                                                                  资产总计  
2023-09-30 00:00:00                               [5516388000000.0000]  
2023-12-31 00:00:00  [5587116000000.0000, 5587116000000.0000, 55871...  
                             资产总计
2023-09-30 00:00:00  5.516388e+12
2023-12-31 00:00:00  5.587116e+12


## FeatureTable

继承自 `WideTable`

数据库数据示例：

![Feature_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Feature_Table_Demo.png)

In [28]:
FT = FDB.getTable("A股证券主表")
print(qs_help(FT))

类型: _FeatureTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: A股证券主表
QSID: 763d905c878358fb0bc484774af32baca6547c02f820a2e9301998c9a255b5c6
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'A股证券主表'
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'FeatureTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: None
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * LookBack(回溯天数): typing.Union[int, float], 默认值 inf, 当前取值: inf
    * PublDTField(公告时点字段): typing.Optional[str], 默认值 None, 用作公告时点的字段名, 默认值 None 表示内部自动判断, 如果非 None, 表示考虑数据的公布时点, 即某个时点所能获取的数据必须保证其在公告时点和截止时点之后, 当前取值: None
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * Operator(算子): typing.Optional[typing.Callable], 默认值 None, 对于单个时点单个 ID 处的数据 apply 的函数 f(x), 其中 x 为 Series, 默认值 None 表示使用 lambda x: x.tolist(),

In [29]:
# FeatureTable 每个时点的数据都一样
DTs = [dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 2)]
IDs = ["000001.SZ"]

FT = FDB.getTable("A股证券主表", args={})
Data = FT.readData(factor_names=["证券简称", "上市日期"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

            证券简称                上市日期
2025-01-01  平安银行  670636800000000000
2025-01-02  平安银行  670636800000000000


## TimeSeriesTable

数据库数据示例：

![TimeSeries_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/TimeSeries_Table_Demo.png)

## NarrowTable

数据库数据示例：

![Narrow_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Narrow_Table_Demo.png)

## MappingTable

数据库数据示例：

![Mapping_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Mapping_Table_Demo.png)

In [30]:
FT = FDB.getTable("公司行业划分表")
print(qs_help(FT))

类型: _MappingTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 公司行业划分表
QSID: 813dcf05c55edb4a74beb91efcbbc558c2023fea130d99df9d8da4e38907bda8
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '公司行业划分表'
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'MappingTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '信息发布日期'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: 
        * 行业划分标准: 3
    * MultiMapping(多重映射): <class 'bool'>, 默认值 False, 是否为高维数据, 即起始时点和 ID 两个维度无法唯一索引单个数据, 默认形成的数据在单个时点单个 ID 处以 list 形式表达, 当前取值: False
    * EndDTField(结束时点字段): <class 'str'>, 无默认值, 用以指示结束填充的时点字段, 默认值 None 表示内部自动判断, 当前取值: '取消日期'
    * EndDTIncluded(包含结束时点): <class 'bool'>, 默认值 True, 结束时点处是否填充数据, 当前取值: False
说明文档:
    基于 SQL 数据库表的映射因子表
    一个字段（参数IDField指定）标识 ID, 一个字段（参数DTField指定）标识起始时点, 一个字段（参数EndDTField指定）标识截止时点, 其余字段为因子


In [31]:
# MappingTable 位于 DTField 和 EndDTField 之间的时点都会填充一样的值
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
IDs = ["600136.SH"]

FT = FDB.getTable("公司行业划分表", args={"AdditionalCondition": {"行业划分标准": "37"}})
Data = FT.readData(factor_names=["一级行业代码", "一级行业名称"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

           一级行业代码 一级行业名称
2018-02-01     22   基础化工
2019-12-03     63     传媒


## ConstituentTable

数据库数据示例：

![Constituent_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Constituent_Table_Demo.png)

In [32]:
FT = FDB.getTable("指数成份")
print(qs_help(FT))

类型: _ConstituentTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 指数成份
QSID: 011ce0799a91e65a44e20287502dd167432023a4547c28c69568268fd942b058
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '指数成份'
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'ConstituentTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '入选日期'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: {}
    * GroupField(类别字段): <class 'str'>, 无默认值, 作为因子名称的字段, 当前取值: '指数内部编码'
    * EndDTField(结束时点字段): <class 'str'>, 无默认值, 用以指示调出成份的时点字段, 当前取值: '剔除日期'
    * EndDTIncluded(包含结束时点): <class 'bool'>, 默认值 False, 结束时点处是否包含在成份中, 当前取值: False
说明文档:
    基于 SQL 数据库表的成份因子表
    一个字段（参数IDField指定）标识 ID, 一个字段（参数DTField指定）标识起始时点, 一个字段（参数EndDTField指定）标识截止时点, 一个字段（参数GroupField指定）标识类别
    因子值是 0-1 变量，1 表示属于 GroupField 指定的类别


In [33]:
# ConstituentTable 位于 DTField 和 EndDTField 之间的时点都会填充 1，其余时点为 0 或者 nan
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
IDs = ["000001.SZ"]

FT = FDB.getTable("指数成份")
Data = FT.readData(factor_names=["000016.SH", "000300.SH"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

           000016.SH 000300.SH
2018-02-01         0         1
2019-12-03         0         1


## FinancialTable

数据库数据示例：

![Financial_Table_Demo](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/Financial_Table_Demo.png)

In [34]:
FT = FDB.getTable("资产负债表_新会计准则")
print(qs_help(FT))

类型: _FinancialTable
模块: QuantStudio.Factor.JYDB
QS 对象类型: 计算节点-因子表
QS 对象名称: 资产负债表_新会计准则
QSID: 667c748c3dd910d8df37ff97a35edd3d5950bdfa42a70dada8a111f8048062fb
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: '资产负债表_新会计准则'
    * TableType(因子表类型): <class 'str'>, 默认值 'WideTable', 只能在 getTable 时传入，因子表创建后不可改变, 用于指明形成的因子表的类型, 当前取值: 'FinancialTable'
    * DTField(时点字段): typing.Optional[str], 默认值 None, 默认 None 表示由内部自动判断. 因子表用于表示时点维度的字段名, 当前取值: '截止日期'
    * AdditionalCondition(附加条件): <class 'dict'>, 默认值 {}, 当前取值: 
        * 公告类别: 20
        * 是否合并: 1
    * ReportDate(报告期): typing.Literal['所有', '定期报告', '年报', '中报', '一季报', '三季报'], 默认值 '所有', 指定原始数据中保留的报告期报告, 当前取值: '所有'
    * CalcType(计算方法): typing.Literal['最新', '单季度', 'TTM'], 默认值 '最新', 财务数据转换成因子数据的方式
        * 最新: 以当前时点能得到的(公告时点在当前时点之前)指定报告期的财务报告的数据值作为当前时点的因子值.
        * 单季度: 以当前时点能得到的(公告时点在当前时点之前)指定报告期的财务报告的数据值计算出的单季度数据作为当前时点的因子值. 比如, 当前时点是 2010 年 8 月 20 日, 指定的报告期是所有, 要计算最新单季度的净利润, 如果某公司在这天以前已经披露了 2010 年中报, 则用 2010 年中报的净利润减去 2010 年一季报的净利润得到二季度

In [35]:
# FinancialTable
DTs = [dt.datetime(2018, 2, 1), dt.datetime(2019, 12, 3)]
IDs = ["000001.SZ"]

FT = FDB.getTable("资产负债表_新会计准则", args={
    "ReportDate": "年报",
    "CalcType": "最新"
})
Data = FT.readData(factor_names=["资产总计"], ids=IDs, dts=DTs).iloc[:, :, 0]
print(Data)

                    资产总计
2018-02-01  2.953434e+12
2019-12-03  3.418592e+12
